# Week 91, Day 2: Security review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w091d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import posixpath

base = "/srv/waymark/exports"
for name in ["ridge.gpx", "../../etc/passwd", "walks/jan.gpx"]:
    print(posixpath.normpath(posixpath.join(base, name)))

> **Key idea.** Model threats before reading code: draw the data flows, mark trust boundaries, and ask STRIDE at each crossing (authentication, validation, audit, disclosure, denial of service, privilege). Limit size, items, text and depth before doing work; rate-limit with a token bucket capped at its capacity. Normalise paths built from input and check they stay inside their folder by components. Send the security headers browsers need.

### Exercise: Trust boundaries

Write `crossings(flows, zones)`: `flows` is a list of `(source, target, data)` tuples and `zones` maps each component to a trust zone (`"internet"`, `"service"` or `"data"`, from least to most trusted). Return a list of `"SOURCE -> TARGET: DATA (from ZONE to ZONE)"` for every flow between different zones **into a more trusted zone**, in the order given.

In [ ]:
def crossings(flows, zones):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("ZONES = {\"browser\": \"internet\", \"partner app\": \"internet\", \"waymark\": \"service\", \"database\": \"data\", \"exports\": \"data\"}\nFLOWS = [(\"browser\", \"waymark\", \"plan request\"), (\"waymark\", \"browser\", \"route\"), (\"partner app\", \"waymark\", \"walk upload\"),\n         (\"waymark\", \"database\", \"saved route\"), (\"database\", \"waymark\", \"walks\"), (\"waymark\", \"exports\", \"gpx file\"),\n         (\"browser\", \"partner app\", \"share link\")]\n\ndef test_crossings():\n    \"where untrusted data enters\"\n    assert crossings(FLOWS, ZONES) == [\n        \"browser -> waymark: plan request (from internet to service)\",\n        \"partner app -> waymark: walk upload (from internet to service)\",\n        \"waymark -> database: saved route (from service to data)\",\n        \"waymark -> exports: gpx file (from service to data)\",\n    ]")

### Exercise: Limits on a request

Write `limit_problems(value, max_depth=5, max_items=100, max_text=200)`, walking a JSON-like value (dicts, lists, strings, numbers) depth first in document order, and returning a list of problems with JSON paths (`$` for the whole value, `.key` for a dict key, `[i]` for a list index): `"PATH: N items (max M)"` for a dict or list with more than `max_items` entries (checked before its contents), `"PATH: N characters (max M)"` for a longer string, and `"PATH: nested too deep (max M)"` for a dict or list deeper than `max_depth` (the outermost container is depth 1), without looking inside it.

In [ ]:
def limit_problems(value, max_depth=5, max_items=100, max_text=200):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fine():\n    \"an ordinary request\"\n    assert limit_problems({\"start\": \"Mill\", \"end\": \"Ridge\", \"avoid\": [\"Farm\"], \"prefs\": {\"steep_limit\": 80}}) == []\n\ndef test_problems():\n    \"too big, too long, too deep\"\n    request = {\"stops\": [\"Mill\"] * 150, \"note\": \"x\" * 250, \"a\": {\"b\": {\"c\": {\"d\": {\"e\": {\"f\": 1}}}}}}\n    assert limit_problems(request) == [\"$.stops: 150 items (max 100)\", \"$.note: 250 characters (max 200)\",\n                                       \"$.a.b.c.d.e: nested too deep (max 5)\"]\n    assert limit_problems([[\"ok\", \"y\" * 9]], max_text=8) == [\"$[0][1]: 9 characters (max 8)\"]\n    assert limit_problems([[[[[[[1]]]]]]], max_depth=3) == [\"$[0][0][0]: nested too deep (max 3)\"]")

### Exercise: A token bucket

Write `TokenBucket(capacity, rate, clock)`: it starts full; `clock()` returns the time in seconds. `allow()` first adds `rate` tokens per second since the last call (never holding more than `capacity`), then spends one token and returns `True` if at least one is available, else returns `False`. Also write `RateLimiter(capacity, rate, clock)` with `allow(client)` keeping one bucket per client.

In [ ]:
class TokenBucket:
    ___

class RateLimiter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Clock:\n    def __init__(self):\n        self.now = 0.0\n    def __call__(self):\n        return self.now\n\ndef test_bucket():\n    \"bursts, then the rate\"\n    clock = Clock()\n    bucket = TokenBucket(5, 2, clock)\n    assert [bucket.allow() for _ in range(7)] == [True] * 5 + [False, False]\n    clock.now = 1.0\n    assert [bucket.allow() for _ in range(3)] == [True, True, False]\n    clock.now = 3600.0\n    assert sum(bucket.allow() for _ in range(100)) == 5\n\ndef test_limiter():\n    \"one bucket each\"\n    clock = Clock()\n    limiter = RateLimiter(2, 1, clock)\n    assert [limiter.allow(\"ada\") for _ in range(3)] == [True, True, False]\n    assert limiter.allow(\"bo\") is True\n    clock.now = 1.5\n    assert limiter.allow(\"ada\") is True and limiter.allow(\"ada\") is False")

### Exercise: Export paths that stay put

Write `export_path(base, name)`: join a user-supplied file `name` to the folder `base` (POSIX paths), normalise it, and return it as a string. Raise `ValueError("unsafe name")` if `name` is empty, contains a NUL character (`"\0"`), is absolute, or the normalised path isn't strictly inside `base` (compare components with `posixpath.commonpath`).

In [ ]:
import posixpath

def export_path(base, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_paths():\n    \"inside the folder\"\n    base = \"/srv/waymark/exports\"\n    assert export_path(base, \"ridge.gpx\") == \"/srv/waymark/exports/ridge.gpx\"\n    assert export_path(base, \"walks/../jan.gpx\") == \"/srv/waymark/exports/jan.gpx\"\n    for bad in [\"\", \"../../etc/passwd\", \"/etc/passwd\", \"../exports-old/x.gpx\", \"ok\\0.gpx\", \".\", \"walks/../..\"]:\n        try:\n            export_path(base, bad)\n        except ValueError as e:\n            assert str(e) == \"unsafe name\"\n        else:\n            raise AssertionError(repr(bad))")

### Exercise: Security headers

Write `header_problems(headers, private=False)`: header names compare without regard to case. Return a list, in this order, of `"missing Content-Security-Policy"`, `"X-Content-Type-Options should be nosniff"` (if missing or any other value), `"missing Strict-Transport-Security"`, and, only when `private` is true, `"Cache-Control should include no-store"`, for each that applies.

In [ ]:
def header_problems(headers, private=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = {\"Content-Security-Policy\": \"default-src 'self'\", \"X-Content-Type-Options\": \"nosniff\",\n        \"Strict-Transport-Security\": \"max-age=31536000\", \"Cache-Control\": \"private, no-store\"}\n\ndef test_headers():\n    \"what a browser needs\"\n    assert header_problems(GOOD, private=True) == []\n    lower = {k.lower(): v for k, v in GOOD.items()}\n    assert header_problems(lower, private=True) == []\n    assert header_problems({\"x-content-type-options\": \"yes\", \"Cache-Control\": \"max-age=60\"}, private=True) == [\n        \"missing Content-Security-Policy\", \"X-Content-Type-Options should be nosniff\",\n        \"missing Strict-Transport-Security\", \"Cache-Control should include no-store\"]\n    assert header_problems({**GOOD, \"Cache-Control\": \"max-age=60\"}) == []")

### Bug hunt: A bucket that filled overnight

After an hour of quiet, this bucket lets a burst of all 100 requests through: it prints `100`, though its capacity is 10. Cap the tokens at the capacity so it prints `10`.

In [ ]:
class Bucket:
    def __init__(self, capacity, rate):
        self.capacity, self.rate = capacity, rate
        self.tokens, self.last = capacity, 0.0

    def allow(self, now):
        self.tokens = self.tokens + (now - self.last) * self.rate
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

bucket = Bucket(10, 1)
print(sum(bucket.allow(3600.0) for _ in range(100)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"never more than the capacity\"\n    expect_output(\"10\")")

### Bug hunt: Inside by the look of it

This check compares text, so a path in the folder `exports-old` passes as if it were inside `exports`: it prints `True`. Compare path components instead, so it prints `False`.

In [ ]:
import posixpath

def inside(base, path):
    return posixpath.normpath(path).startswith(posixpath.normpath(base))

print(inside("/srv/waymark/exports", "/srv/waymark/exports-old/walks.csv"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"components, not characters\"\n    expect_output(\"False\")")

### Puzzle: Spending a token

Arrange `allow`: refill for the time passed, capped at the capacity, then spend a token if there is one. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the STRIDE letter for "someone denies changing a saved route, and there's no record"
q = ___   # the HTTP status for a request refused by a rate limiter
r = ___   # tokens a bucket of capacity 5 and rate 2 per second holds after 1 hour idle
s = ___   # posixpath.normpath("/srv/exports/a/../../b")
t = ___   # True or False: the header X-Content-Type-Options should be "nosniff"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"R\" and q == 429 and r == 5, \"p, q and r\"\n    assert s == \"/srv/b\" and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a threat list from the data flows

Write `threats(components, flows)`. `components` maps a name to `{"zone": ..., "kind": ...}`, with zones `"internet"`, `"service"` and `"data"` and kinds `"external"`, `"process"` and `"store"`. Each flow is a dict with `from`, `to`, `data`, `checks` (a list of strings) and optional flags `secret`, `writes` and `admin` (false if missing). Apply these rules to each flow:

- **into** the service from the internet: `Spoofing` if `"auth"` isn't checked (`"no authentication"`), `Tampering` if `"validate"` isn't (`"input not validated"`), `Denial of service` if `"rate limit"` isn't (`"not rate limited"`), and `Elevation of privilege` if it's an `admin` flow without `"role check"` (`"admin action without a role check"`);
- **out to** the internet: `Information disclosure` if it's `secret` without `"tls"` (`"sent without TLS"`);
- **to a store**: `Repudiation` if it `writes` without `"audit"` (`"changes not audited"`).

Return a list of `"CATEGORY: FROM -> TO (DATA): PROBLEM"`, sorted in STRIDE order (Spoofing, Tampering, Repudiation, Information disclosure, Denial of service, Elevation of privilege), then in flow order.

In [ ]:
def threats(components, flows):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("COMPONENTS = {\"browser\": {\"zone\": \"internet\", \"kind\": \"external\"}, \"partner app\": {\"zone\": \"internet\", \"kind\": \"external\"},\n              \"waymark\": {\"zone\": \"service\", \"kind\": \"process\"}, \"database\": {\"zone\": \"data\", \"kind\": \"store\"},\n              \"exports\": {\"zone\": \"data\", \"kind\": \"store\"}}\nFLOWS = [\n    {\"from\": \"browser\", \"to\": \"waymark\", \"data\": \"plan request\", \"checks\": [\"auth\", \"validate\", \"rate limit\"]},\n    {\"from\": \"partner app\", \"to\": \"waymark\", \"data\": \"walk upload\", \"checks\": [\"auth\"]},\n    {\"from\": \"browser\", \"to\": \"waymark\", \"data\": \"delete user\", \"checks\": [\"auth\", \"validate\", \"rate limit\"], \"admin\": True},\n    {\"from\": \"waymark\", \"to\": \"browser\", \"data\": \"share page\", \"checks\": [], \"secret\": True},\n    {\"from\": \"waymark\", \"to\": \"browser\", \"data\": \"route\", \"checks\": [\"tls\"], \"secret\": True},\n    {\"from\": \"waymark\", \"to\": \"database\", \"data\": \"saved route\", \"checks\": [\"audit\"], \"writes\": True},\n    {\"from\": \"waymark\", \"to\": \"exports\", \"data\": \"gpx file\", \"checks\": [], \"writes\": True},\n    {\"from\": \"database\", \"to\": \"waymark\", \"data\": \"walks\", \"checks\": []},\n]\n\ndef test_threats():\n    \"STRIDE at the boundaries\"\n    assert threats(COMPONENTS, FLOWS) == [\n        \"Tampering: partner app -> waymark (walk upload): input not validated\",\n        \"Repudiation: waymark -> exports (gpx file): changes not audited\",\n        \"Information disclosure: waymark -> browser (share page): sent without TLS\",\n        \"Denial of service: partner app -> waymark (walk upload): not rate limited\",\n        \"Elevation of privilege: browser -> waymark (delete user): admin action without a role check\",\n    ]\n\ndef test_unauthenticated():\n    \"every check missing\"\n    flow = {\"from\": \"browser\", \"to\": \"waymark\", \"data\": \"x\", \"checks\": [], \"admin\": True}\n    assert [t.split(\":\")[0] for t in threats(COMPONENTS, [flow])] == [\"Spoofing\", \"Tampering\", \"Denial of service\", \"Elevation of privilege\"]\n    assert threats(COMPONENTS, []) == []")

## Warm-up

The second name walks out of the export folder. Any path built from user input needs the same suspicion as a SQL query built from it: check where it **ends up**, not what it looks like.

## A threat model

A threat model answers four questions (from Adam Shostack's *Threat Modeling*): what are we building, what can go wrong, what are we going to do about it, and did we do a good job?

Start with a **data-flow diagram**: the **external** parties (a walker's browser, another app using the API), the **processes** (the Waymark service), the **stores** (the database, the export folder), and the **flows** between them. Then draw the **trust boundaries**: lines where data passes from someone you don't control to something you do. Every flow across a boundary is where attacks arrive.

At each crossing ask the six **STRIDE** questions (devised at Microsoft by Loren Kohnfelder and Praerit Garg):

- **S**poofing: can someone pretend to be another user? (Authentication.)
- **T**ampering: can they change data they shouldn't, or send input that breaks things? (Validation.)
- **R**epudiation: could someone deny doing something, with no record to show otherwise? (Audit logs.)
- **I**nformation disclosure: can data leak, in transit or in errors? (Encryption, careful errors.)
- **D**enial of service: can one client exhaust the service? (Limits, rate limiting.)
- **E**levation of privilege: can someone do what only an administrator should? (Role checks.)

## Limits and rate limiting

**Limits** stop a single request costing too much: a maximum body size, a maximum number of items in a list (a route with 100,000 stops), a maximum string length, and a maximum **nesting depth** (JSON nested ten thousand levels deep can crash a parser that recurses). Check them before doing any real work.

**Rate limiting** stops a single client sending too many requests. A **token bucket** holds up to `capacity` tokens and gains `rate` tokens per second; each request spends one, and a request with no token left is refused (status 429). It allows short bursts up to the capacity while holding the long-run rate down. The bucket must never hold **more** than its capacity, however long it's been idle.

## Paths and headers

A file name from a user (an export called `ridge.gpx`) is joined to a folder, so `../` can climb out of it. Normalise the joined path and check that it's still **inside** the folder, comparing path components with `posixpath.commonpath` (or `Path.is_relative_to`), not strings: `/srv/exports-old` starts with the text `/srv/exports` but isn't inside it.

Pages a browser shows need a few **security headers**: `Content-Security-Policy` (which scripts may run), `X-Content-Type-Options: nosniff` (don't guess file types), `Strict-Transport-Security` (always use HTTPS), and `Cache-Control: no-store` on pages with private data, such as an opened share.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Draw your data-flow diagram** (paper is fine): externals, processes, stores, flows, and the trust boundaries.
2. **STRIDE at every crossing**: write each threat you find, what stops it now, and what you'll do about the ones nothing stops.
3. **Harden** the top three: limits, a rate limiter, safe paths, headers, or whatever your model shows.
4. **Scan the history** for committed secrets with a free secret scanner, and rotate anything it finds.

## Before you go

Tomorrow is about what happens when things go wrong anyway: errors users can act on, and logs you can act on.